# 第4章：数据处理器 (DataHandler)

## 本章学习目标

- 理解 DataHandler 的设计理念
- 掌握数据预处理流水线
- 熟练使用内置处理器
- 能够构建自定义 DataHandler

---

## 4.1 DataHandler 概述

DataHandler 是 qlib 数据处理的核心组件，负责：

1. **数据加载**：从 Provider 获取原始数据
2. **特征生成**：计算衍生特征（如 Alpha 因子）
3. **数据预处理**：清洗、标准化、去极值等
4. **标签生成**：构建预测目标

### 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                    DataHandler 架构                         │
├─────────────────────────────────────────────────────────────┤
│  ┌─────────────────────────────────────────────────────┐   │
│  │                   DataHandlerLP                      │   │
│  │            (Learnable Processor Pipeline)            │   │
│  └─────────────────────────────────────────────────────┘   │
│                          ↓                                  │
│  ┌───────────┐  ┌───────────┐  ┌───────────┐              │
│  │ Processor │→ │ Processor │→ │ Processor │   ...        │
│  │  (缺失值) │  │  (标准化) │  │  (去极值) │              │
│  └───────────┘  └───────────┘  └───────────┘              │
│                          ↓                                  │
│                   处理后的数据                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
import qlib
from qlib.data.dataset.handler import DataHandler, DataHandlerLP
from qlib.data.dataset.processor import Processor
from qlib.contrib.data.handler import Alpha158, Alpha360
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 4.2 DataHandler 基类

### 4.2.1 DataHandler 类层次

```
DataHandler (基类)
    │
    ├── DataHandlerLP (带预处理流水线)
    │       │
    │       ├── Alpha158 (158个因子)
    │       └── Alpha360 (360个因子)
    │
    └── 其他自定义 DataHandler
```

In [ ]:
# 查看 DataHandler 基类结构
from qlib.data.dataset.handler import DataHandler
import inspect

# 打印 DataHandler 的主要方法
print("DataHandler 主要方法:")
for name, method in inspect.getmembers(DataHandler, predicate=inspect.isfunction):
    if not name.startswith("_"):
        print(f"  - {name}")

In [ ]:
# 查看 DataHandlerLP 的主要方法
from qlib.data.dataset.handler import DataHandlerLP

print("DataHandlerLP 主要方法:")
for name, method in inspect.getmembers(DataHandlerLP, predicate=inspect.isfunction):
    if not name.startswith("_"):
        print(f"  - {name}")

## 4.3 使用内置 DataHandler

### 4.3.1 Alpha158 特征集

In [ ]:
# 创建 Alpha158 DataHandler
handler = Alpha158(
    instruments="csi300",
    start_time="2020-01-01",
    end_time="2022-12-31",
    freq="day",
)

print(f"Handler 类型: {type(handler).__name__}")
print(f"开始时间: {handler.start_time}")
print(f"结束时间: {handler.end_time}")

In [ ]:
# 获取处理后的特征数据
# fetch() 方法返回处理后的 DataFrame
df_features = handler.fetch()

print(f"特征数据形状: {df_features.shape}")
print(f"\n特征列数: {len(df_features.columns)}")
print(f"\n前 10 个特征列:")
print(df_features.columns[:10].tolist())

In [ ]:
# 查看特征数据
df_features.head()

In [ ]:
# 查看数据类型和缺失情况
print("特征数据统计信息:")
print(f"  总行数: {len(df_features)}")
print(f"  总列数: {len(df_features.columns)}")
print(f"  缺失值总数: {df_features.isna().sum().sum()}")
print(f"  缺失比例: {df_features.isna().sum().sum() / df_features.size:.4%}")

### 4.3.2 配置 DataHandler

In [ ]:
# 带预处理器的 Alpha158
from qlib.data.dataset.processor import ZScoreNorm, DropnaProcessor, ProcessInf

# 创建带预处理的 handler
handler_with_processor = Alpha158(
    instruments="csi300",
    start_time="2020-01-01",
    end_time="2022-12-31",
    freq="day",
    infer_processors=[
        {"class": "DropnaProcessor", "fields_group": "feature"},
        {"class": "ZScoreNorm", "fields_group": "feature"},
        {"class": "ProcessInf", "fields_group": "feature"},
    ],
)

print("带预处理的 Alpha158 创建成功")

In [ ]:
# 获取处理后的数据
df_processed = handler_with_processor.fetch()

print(f"处理后数据形状: {df_processed.shape}")
df_processed.head()

In [ ]:
# 对比处理前后的数据分布
import matplotlib.pyplot as plt

# 选择一个特征进行对比
feature_col = df_features.columns[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 处理前
axes[0].hist(df_features[feature_col].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title(f'处理前: {feature_col}')
axes[0].set_xlabel('值')
axes[0].set_ylabel('频数')

# 处理后
axes[1].hist(df_processed[feature_col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title(f'处理后: {feature_col}')
axes[1].set_xlabel('值')
axes[1].set_ylabel('频数')

plt.tight_layout()
plt.show()

# 统计对比
print(f"\n处理前统计:")
print(f"  均值: {df_features[feature_col].mean():.4f}")
print(f"  标准差: {df_features[feature_col].std():.4f}")
print(f"  缺失值: {df_features[feature_col].isna().sum()}")

print(f"\n处理后统计:")
print(f"  均值: {df_processed[feature_col].mean():.4f}")
print(f"  标准差: {df_processed[feature_col].std():.4f}")
print(f"  缺失值: {df_processed[feature_col].isna().sum()}")

## 4.4 数据预处理器体系

Qlib 提供了多种内置的数据预处理器：

| 处理器 | 功能 | 参数 |
|--------|------|------|
| `DropnaProcessor` | 删除缺失值 | `fields_group` |
| `DropnaLabel` | 删除标签缺失值 | `fields_group` |
| `ZScoreNorm` | Z-score 标准化 | `fields_group`, `fit_start_time`, `fit_end_time` |
| `MinMaxNorm` | Min-Max 归一化 | `fields_group` |
| `ProcessInf` | 处理无穷值 | `fields_group` |
| `Fillna` | 填充缺失值 | `fields_group`, `fill_value` |

In [ ]:
# 预处理器详解
from qlib.data.dataset.processor import (
    DropnaProcessor,
    ZScoreNorm,
    MinMaxNorm,
    ProcessInf,
    Fillna,
)

# 创建测试数据
test_data = pd.DataFrame({
    'feature_1': [1, 2, np.nan, 4, 5, 100, np.inf, -np.inf],
    'feature_2': [0.1, 0.2, 0.3, np.nan, 0.5, 0.6, 0.7, 0.8],
})

print("原始数据:")
print(test_data)

In [ ]:
# 处理无穷值
processor_inf = ProcessInf(fields_group="")
data_no_inf = test_data.copy()
processor_inf(data_no_inf)

print("处理无穷值后:")
print(data_no_inf)

In [ ]:
# 填充缺失值
processor_fillna = Fillna(fields_group="", fill_value=0)
data_filled = test_data.copy()
processor_fillna(data_filled)

print("填充缺失值后 (fill_value=0):")
print(data_filled)

In [ ]:
# Z-score 标准化
processor_zscore = ZScoreNorm(fields_group="")
data_zscore = test_data.copy()
# 先处理无穷值和缺失值
ProcessInf(fields_group="")(data_zscore)
Fillna(fields_group="", fill_value=0)(data_zscore)
# 再标准化
processor_zscore(data_zscore)

print("Z-score 标准化后:")
print(data_zscore)

In [ ]:
# Min-Max 归一化
processor_minmax = MinMaxNorm(fields_group="")
data_minmax = test_data.copy()
# 先处理无穷值和缺失值
ProcessInf(fields_group="")(data_minmax)
Fillna(fields_group="", fill_value=0)(data_minmax)
# 再归一化
processor_minmax(data_minmax)

print("Min-Max 归一化后:")
print(data_minmax)

## 4.5 构建自定义 DataHandler

### 4.5.1 简单自定义 DataHandler

In [ ]:
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.data import D

class SimpleHandler(DataHandlerLP):
    """简单的自定义 DataHandler"""
    
    def __init__(
        self,
        instruments="csi300",
        start_time=None,
        end_time=None,
        freq="day",
        infer_processors=[],
        learn_processors=[],
        **kwargs
    ):
        # 基础配置
        self.instruments = instruments
        self.start_time = start_time
        self.end_time = end_time
        self.freq = freq
        
        # 定义特征字段
        # 这里我们使用表达式定义一些简单的技术指标
        self.fields = [
            # 价格数据
            "$close",
            "$open",
            "$high",
            "$low",
            "$volume",
            # 移动平均
            "Mean($close, 5)",
            "Mean($close, 10)",
            "Mean($close, 20)",
            # 动量
            "$close / Ref($close, 5) - 1",
            "$close / Ref($close, 10) - 1",
            # 波动率
            "Std($close, 10)",
            "Std($close, 20)",
            # 成交量变化
            "$volume / Mean($volume, 5)",
        ]
        
        super().__init__(
            instruments=instruments,
            start_time=start_time,
            end_time=end_time,
            freq=freq,
            infer_processors=infer_processors,
            learn_processors=learn_processors,
            **kwargs
        )
    
    def setup_data(self, **kwargs):
        """设置数据加载"""
        # 调用父类方法
        super().setup_data(**kwargs)

# 创建自定义 handler
custom_handler = SimpleHandler(
    instruments="csi300",
    start_time="2022-01-01",
    end_time="2022-12-31",
)

print(f"自定义 Handler 创建成功")
print(f"特征数量: {len(custom_handler.fields)}")

In [ ]:
# 获取数据
df_custom = custom_handler.fetch()

print(f"数据形状: {df_custom.shape}")
print(f"\n特征列:")
print(df_custom.columns.tolist())
df_custom.head()

### 4.5.2 带预处理的 DataHandler

In [ ]:
class ProcessedHandler(DataHandlerLP):
    """带预处理流水线的 DataHandler"""
    
    def __init__(
        self,
        instruments="csi300",
        start_time=None,
        end_time=None,
        freq="day",
        **kwargs
    ):
        # 定义特征
        self.fields = [
            "$close",
            "Mean($close, 5)",
            "Mean($close, 20)",
            "$close / Ref($close, 5) - 1",
            "Std($close, 20)",
        ]
        
        # 定义预处理流水线
        infer_processors = [
            # 1. 处理无穷值
            {"class": "ProcessInf", "fields_group": "feature"},
            # 2. 填充缺失值
            {"class": "Fillna", "fields_group": "feature", "fill_value": 0},
            # 3. Z-score 标准化
            {"class": "ZScoreNorm", "fields_group": "feature"},
        ]
        
        super().__init__(
            instruments=instruments,
            start_time=start_time,
            end_time=end_time,
            freq=freq,
            infer_processors=infer_processors,
            **kwargs
        )

# 创建带预处理的 handler
processed_handler = ProcessedHandler(
    instruments="csi300",
    start_time="2022-01-01",
    end_time="2022-12-31",
)

print("带预处理的 Handler 创建成功")

In [ ]:
# 获取处理后的数据
df_processed = processed_handler.fetch()

print(f"处理后数据形状: {df_processed.shape}")
df_processed.head()

In [ ]:
# 检查处理效果
print("处理后数据统计:")
for col in df_processed.columns:
    print(f"\n{col}:")
    print(f"  均值: {df_processed[col].mean():.6f}")
    print(f"  标准差: {df_processed[col].std():.6f}")
    print(f"  缺失值: {df_processed[col].isna().sum()}")

## 4.6 数据集 (Dataset) 与 DataHandler

DataHandler 通常与 Dataset 配合使用，Dataset 负责管理数据分割和采样。

In [ ]:
from qlib.data.dataset import DatasetH

# 创建 Dataset
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2018-01-01",
            "end_time": "2022-12-31",
            "fit_start_time": "2018-01-01",
            "fit_end_time": "2020-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2018-01-01", "2020-12-31"),
        "valid": ("2021-01-01", "2021-06-30"),
        "test": ("2021-07-01", "2022-12-31"),
    },
)

print("Dataset 创建成功")

In [ ]:
# 获取不同分割的数据
train_data = dataset.prepare("train")
valid_data = dataset.prepare("valid")
test_data = dataset.prepare("test")

print(f"训练集形状: {train_data.shape}")
print(f"验证集形状: {valid_data.shape}")
print(f"测试集形状: {test_data.shape}")

In [ ]:
# 查看训练数据
train_data.head()

## 4.7 实践练习

### 练习目标

1. 实现 3 个数据预处理器：缺失值填充、标准化、去极值
2. 构建 DataHandler 组合处理器
3. 对比不同预处理方法的效果
4. 分析处理前后数据分布变化

In [ ]:
# 练习1: 实现一个自定义预处理器
# 功能：将超过 3 倍标准差的值替换为边界值（Winsorization）

from qlib.data.dataset.processor import Processor

class WinsorizeProcessor(Processor):
    """去极值处理器"""
    
    def __init__(self, fields_group="feature", n_std=3):
        super().__init__(fields_group=fields_group)
        self.n_std = n_std
        self.lower_bound = None
        self.upper_bound = None
    
    def fit(self, df):
        """拟合处理器，计算边界值"""
        mean = df.mean()
        std = df.std()
        self.lower_bound = mean - self.n_std * std
        self.upper_bound = mean + self.n_std * std
        return self
    
    def __call__(self, df):
        """应用处理"""
        if self.lower_bound is None:
            self.fit(df)
        
        # Clip to bounds
        for col in df.columns:
            df[col] = df[col].clip(
                lower=self.lower_bound[col],
                upper=self.upper_bound[col]
            )

# 测试自定义处理器
test_df = pd.DataFrame({
    'feature': [1, 2, 3, 4, 5, 100, -50, 6, 7, 8],
})

print("原始数据:")
print(test_df)

processor = WinsorizeProcessor(n_std=2)
processor(test_df)

print("\n处理后:")
print(test_df)

In [ ]:
# 练习2: 构建包含自定义处理器的 DataHandler

class MyHandler(DataHandlerLP):
    """自定义 DataHandler"""
    
    def __init__(self, instruments="csi300", start_time=None, end_time=None, **kwargs):
        self.fields = [
            "$close",
            "$volume",
            "Mean($close, 5)",
            "Mean($close, 20)",
        ]
        
        # 你的代码：定义预处理流水线
        # 包含：处理无穷值、填充缺失值、去极值、标准化
        infer_processors = [
            # 添加你的处理器配置
        ]
        
        super().__init__(
            instruments=instruments,
            start_time=start_time,
            end_time=end_time,
            infer_processors=infer_processors,
            **kwargs
        )

# 创建并测试
# your_handler = MyHandler(instruments="csi300", start_time="2022-01-01", end_time="2022-12-31")

In [ ]:
# 练习3: 对比不同预处理方法的效果

# 创建三个不同的 handler
# 1. 无预处理
# 2. 仅标准化
# 3. 完整预处理（填充 + 去极值 + 标准化）

# 你的代码



# 提示：
# handler_1 = SimpleHandler(..., infer_processors=[])
# handler_2 = SimpleHandler(..., infer_processors=[{"class": "ZScoreNorm", "fields_group": "feature"}])
# handler_3 = SimpleHandler(..., infer_processors=[...])

## 4.8 本章小结

本章我们学习了：

1. **DataHandler 架构**：
   - `DataHandler` 基类
   - `DataHandlerLP` 带预处理流水线
   - `Alpha158` / `Alpha360` 内置特征集

2. **预处理器体系**：
   - `DropnaProcessor` - 删除缺失值
   - `ZScoreNorm` - Z-score 标准化
   - `MinMaxNorm` - Min-Max 归一化
   - `ProcessInf` - 处理无穷值
   - `Fillna` - 填充缺失值

3. **自定义 DataHandler**：
   - 继承 `DataHandlerLP`
   - 定义 `fields` 特征表达式
   - 配置 `infer_processors` 预处理流水线

### 关键 API 速查

```python
# 创建 Alpha158
handler = Alpha158(
    instruments="csi300",
    start_time="2020-01-01",
    end_time="2022-12-31",
)

# 获取数据
df = handler.fetch()

# 配置处理器
handler = Alpha158(
    ...,
    infer_processors=[
        {"class": "DropnaProcessor", "fields_group": "feature"},
        {"class": "ZScoreNorm", "fields_group": "feature"},
    ],
)
```

### 下一章预告

下一章我们将深入学习特征工程与 Alpha 因子，包括：
- Alpha158 特征集详解
- 表达式引擎深入使用
- 开发自定义因子